<a href="https://colab.research.google.com/github/SunSpot-Tech/Flyrank_Internship/blob/main/work/notebooks/%20%20w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SunSpot-Tech/Flyrank_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import duckdb, os, pandas as pd
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{os.environ["HF_TOKEN"]}'
);
""")

FACT = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet"
DIM  = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

feature_frame = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        DATE '2026-03-16' - d.content_updated_date AS days_since_last_update,
        SUM(CASE WHEN f.report_date < DATE '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS impressions_first_half,
        SUM(CASE WHEN f.report_date < DATE '2026-03-16' THEN f.gsc_clicks ELSE 0 END) AS clicks_first_half,
        AVG(CASE WHEN f.report_date < DATE '2026-03-16' THEN f.gsc_avg_position END) AS avg_position_first_half,
        SUM(CASE WHEN f.report_date >= DATE '2026-03-16' THEN f.gsc_clicks ELSE 0 END) AS clicks_second_half
    FROM read_parquet('{FACT}', hive_partitioning=1) f
    JOIN read_parquet('{DIM}') d ON f.content_hash_id = d.content_hash_id
    WHERE f.month = '2026-03'
      AND f.gsc_data_available IS TRUE
    GROUP BY f.content_hash_id, f.client_hash_id, d.content_updated_date
""").df()

# THE FIX: content_updated_date is a current snapshot (frozen July 2026),
# not point-in-time as of March 2026 — so negative "days since update"
# means the update happened AFTER our March cutoff and can't be trusted
# as a March-knowable fact. Treat those as missing, not as "very fresh."
feature_frame['days_since_last_update'] = feature_frame['days_since_last_update'].where(
    feature_frame['days_since_last_update'] >= 0, pd.NA
)

feature_frame['is_declining'] = (feature_frame.clicks_second_half < feature_frame.clicks_first_half).astype(int)
feature_frame['ctr_first_half'] = feature_frame.clicks_first_half / feature_frame.impressions_first_half.replace(0, pd.NA)

print(feature_frame.shape)
print("Rows with valid (non-negative) days_since_last_update:", feature_frame['days_since_last_update'].notna().sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(176738, 9)
Rows with valid (non-negative) days_since_last_update: 27935


In [ ]:
bins = [-1, 30, 90, 180, 365, 10_000]
labels = ['0-30d', '31-90d', '91-180d', '181-365d', '365d+']
feature_frame['staleness_bucket'] = pd.cut(feature_frame.days_since_last_update, bins=bins, labels=labels)

staleness_table = feature_frame.groupby('staleness_bucket', observed=True).agg(
    n=('is_declining', 'size'),
    decline_rate=('is_declining', 'mean')
).round(3)
print(staleness_table)

                      n  decline_rate
staleness_bucket                     
0-30d             26003         0.193
31-90d              346         0.052
91-180d            1404         0.058
181-365d            182         0.016


In [ ]:
pos_bins = [0, 3, 6, 10, 20, 1000]
pos_labels = ['1-3', '4-6', '7-10', '11-20', '20+']
feature_frame['position_tier'] = pd.cut(feature_frame.avg_position_first_half, bins=pos_bins, labels=pos_labels)

tier_avg = feature_frame.groupby('position_tier', observed=True)['ctr_first_half'].mean()
feature_frame['tier_avg_ctr'] = feature_frame['position_tier'].map(tier_avg).astype(float)
feature_frame['ctr_gap'] = feature_frame['ctr_first_half'] < feature_frame['tier_avg_ctr']

ctr_gap_table = feature_frame.groupby(['position_tier', 'ctr_gap'], observed=True).agg(
    n=('is_declining', 'size'),
    decline_rate=('is_declining', 'mean')
).round(3)
print(ctr_gap_table)

                           n  decline_rate
position_tier ctr_gap                     
1-3           False     1586         0.642
              True     14665         0.204
4-6           False     5804         0.623
              True     29563         0.183
7-10          False     5625         0.658
              True     29008         0.086
11-20         False     5824         0.566
              True     21089         0.078
20+           False     4079         0.708
              True     33432         0.055


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal check results:**
- **Staleness: OPPOSITE.** Decline rate falls as staleness increases
  (19.3% at 0-30d down to 1.6% at 181-365d) — the opposite of the
  "stale pages decline more" hypothesis behind refresh flags.
- **CTR gap vs. tier: OPPOSITE.** In every position tier, pages below
  their tier's average CTR show a *lower* decline rate than pages at or
  above it (e.g. tier 20+: 5.5% vs 70.8%) — the opposite of "a CTR gap
  causes decline."

**What this means for my rule:** neither signal predicts decline risk
in the direction I assumed. Rather than force a "predict decline" framing
onto signals that don't support it, I'm reframing the rule honestly as
an **opportunity/attention queue**, not a decline-risk queue: pages that
are stale AND still visible are worth a human look regardless of whether
they're currently declining, because visibility without recent attention
is itself worth reviewing. I no longer claim this rule predicts decline —
only that it flags pages meeting specific, defensible criteria for review.

**Reason codes this rule outputs:**
- `STALE_AND_VISIBLE` — valid staleness ≥180 days and impressions ≥500
- `CTR_GAP_VS_TIER` — CTR below its own position tier's average
- `STALE_AND_CTR_GAP` — both conditions true

**Data quality fix applied:** `content_updated_date` is a snapshot
(frozen July 2026), not point-in-time as of March — negative "days
since update" values are excluded rather than misread as freshness.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
def assign_reason(row):
    is_stale = pd.notna(row['days_since_last_update']) and row['days_since_last_update'] >= 180
    is_visible = row['impressions_first_half'] >= 500
    has_ctr_gap = row['ctr_gap']

    if is_stale and is_visible and has_ctr_gap:
        return 'STALE_AND_CTR_GAP'
    elif is_stale and is_visible:
        return 'STALE_AND_VISIBLE'
    elif has_ctr_gap:
        return 'CTR_GAP_VS_TIER'
    else:
        return None

feature_frame['reason_code'] = feature_frame.apply(assign_reason, axis=1)
feature_frame['action_label'] = 'review_for_refresh'

feature_frame['baseline_score'] = feature_frame['impressions_first_half'] * (
    feature_frame['reason_code'].eq('STALE_AND_CTR_GAP').astype(int) * 2
    + feature_frame['reason_code'].isin(['STALE_AND_VISIBLE', 'CTR_GAP_VS_TIER']).astype(int)
)

queue = feature_frame[feature_frame['reason_code'].notna()].sort_values('baseline_score', ascending=False)

os.makedirs('work/outputs', exist_ok=True)
queue[['content_hash_id', 'client_hash_id', 'baseline_score', 'reason_code',
       'action_label', 'days_since_last_update', 'impressions_first_half',
       'clicks_first_half', 'avg_position_first_half', 'ctr_gap', 'is_declining']].to_csv(
    'work/outputs/baseline_action_score.csv', index=False
)
print(feature_frame['days_since_last_update'].min(), feature_frame['days_since_last_update'].max())
print(feature_frame['days_since_last_update'].isna().sum())

print(f"Queue written: {len(queue)} rows")
print(queue['reason_code'].value_counts())
print(queue.head(20).to_string())

0.0 288.0
148803
Queue written: 127757 rows
reason_code
CTR_GAP_VS_TIER      127754
STALE_AND_CTR_GAP         3
Name: count, dtype: int64
                 content_hash_id           client_hash_id  days_since_last_update  impressions_first_half  clicks_first_half  avg_position_first_half  clicks_second_half  is_declining ctr_first_half staleness_bucket position_tier  tier_avg_ctr  ctr_gap      reason_code        action_label  baseline_score
104824  content_e8a52cf3d5988c07  client_23a62021009f63c4                     NaN                143173.0              353.0                16.018687               316.0             1       0.002466              NaN         11-20      0.003337     True  CTR_GAP_VS_TIER  review_for_refresh        143173.0
154780  content_ec2e0346994fb5a5  client_e547b89c05043229                     NaN                132811.0              843.0                 2.441784               637.0             1       0.006347              NaN           1-3      0.009315     Tr

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Note on the mix:** 127,754 of 127,757 flagged rows carry
`CTR_GAP_VS_TIER` alone, and only 3 carry `STALE_AND_CTR_GAP` — meaning
almost no page in this dataset has *both* a large-enough CTR gap and a
valid (non-negative), genuinely stale `days_since_last_update`. Given
Week 4's OPPOSITE staleness finding (fresher pages decline *more*), this
scarcity actually makes sense: pages that are old enough to be "stale"
are disproportionately the same pages that have already stabilized and
are *not* showing CTR gaps or declining. The two conditions rarely
co-occur in this data.

1. **content_e8a52cf3d5988c07** — reason: CTR_GAP_VS_TIER (CTR 0.0025 vs
   tier avg 0.0033, tier 11-20). declining=1. Wrong if: tier 11-20 has
   few comparable pages, making its average unstable.
2. **content_ec2e0346994fb5a5** — CTR 0.0063 vs avg 0.0093, tier 1-3.
   declining=1. Wrong if: top-tier pages naturally have more CTR variance
   page-to-page due to brand/title differences unrelated to quality.
3. **content_36e53e9c707674fc** — CTR 0.0010 vs avg 0.0021, tier 20+.
   declining=0 (rule flagged but page didn't decline — expected, given
   OPPOSITE finding that low-CTR pages decline less).
4. **content_7172a7fad43f0998** — CTR 0.0045 vs avg 0.0056, tier 4-6.
   declining=1.
5. **content_b99ea6861864dea5** — CTR 0.0020 vs avg 0.0056, sizable gap.
   declining=1.
6. **content_7c6373141eae744a** — CTR 0.0006 vs avg 0.0056, large gap.
   declining=1.
7. **content_3df3f32f3fd58dea** — CTR 0.0014 vs avg 0.0021, tier 20+.
   declining=1.
8. **content_9c057b66c30a3abb** — the only row with a *valid* staleness
   value (19 days) — but 0 clicks, so CTR=0 trivially. declining=0.
   **Weak pick**: zero-click CTR is a degenerate case, not a real gap.
9. **content_acbcc847f8996314** — CTR 0.0016 vs avg 0.0056. declining=1.
10. **content_471d9cabce329a66** — CTR 0.0025 vs avg 0.0056. declining=1.
11. **content_fd2117c2c6790e4b** — CTR 0.0026 vs avg 0.0056. declining=0.
12. **content_34a70fea29d15f24** — CTR 0.0002 vs avg 0.0093, huge gap,
    tier 1-3. declining=0 — rule wrong despite a strong-looking gap.
13. **content_5e1c049f62e33b11** — CTR 0.0015 vs avg 0.0033. declining=1.
14. **content_82e35c4845e6c391** — CTR 0.0004 vs avg 0.0033. declining=0.
15. **content_df47d1b976106de4** — CTR 0.0013 vs avg 0.0021. declining=1.
16. **content_559cdd76da9306de** — CTR 0.00003 vs avg 0.0021, near-zero.
    declining=1. High confidence given gap size.
17. **content_987d251ee617d9c6** — CTR 0.0078 vs avg 0.0093, tier 1-3,
    small gap. declining=0. **Weak pick**: gap is proportionally small.
18. **content_bdf60c86117079be** — CTR 0.0001 vs avg 0.0021. declining=1.
19. **content_3b6e4c8d9a0a5c9c** — CTR 0.0033 vs avg 0.0056. declining=1.
20. **content_8e1334d6356668e3** — CTR 0.00002 vs avg 0.0056, near-zero.
    declining=1. High confidence.

**Precision@20 (rows where declining=1 matched): 14/20 = 0.70** — same
directional strength as before the fix, since the fix mainly affected
which staleness label attaches, not which rows top the CTR-driven ranking.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak Picks + Leakage Check

**Weak picks:**
- Row 8 (`content_9c057b66c30a3abb`): 0 clicks makes CTR trivially 0,
  flagged as a "gap" by default rather than a genuine signal.
- Row 17 (`content_987d251ee617d9c6`): CTR 0.0078 vs tier avg 0.0093 —
  a small, arguably negligible gap relative to tier variance.

**Leakage check:**
- `baseline_score` and `reason_code` are built only from
  `impressions_first_half`, `clicks_first_half`, `avg_position_first_half`,
  and the corrected `days_since_last_update` — all prior-window (first
  half of March) signals.
- `clicks_second_half` and `is_declining` exist in `feature_frame` for
  evaluation only, never referenced by `assign_reason` or the score
  formula — confirmed by re-reading the code.
- **Data quality issue found and fixed:** `content_updated_date` is a
  snapshot (frozen July 2026), not point-in-time as of March — negative
  "days since update" values (148,803 of 176,738 rows) are now excluded
  from the staleness check as missing, rather than misread as extreme
  freshness. Only 27,935 rows carry a trustworthy staleness value.
- **Signal-level finding, not leakage but worth restating:** both
  staleness and CTR-gap turned out OPPOSITE of their hypothesized
  direction (Week 4) — decline is more common in fresh, good-CTR pages,
  not stale, low-CTR ones. My rule is reframed as an attention/opportunity
  queue rather than a decline-prediction queue, and no claim of predicting
  decline is made anywhere in this notebook.
- No product decision flags (`priority_score`, `health_score`,
  `action_type`) exist in this schema, so nothing of that kind could leak.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.